# Sign classifier: keypoint positions only

One shared classifier for comparing pose estimators (SMPLest-X, Sapiens, MediaPipe, ...) on isolated ASL signs. Every model gets the **same** splits, preprocessing, network and training settings, so differences in accuracy come from the keypoints.

**How to use**
1. Save your keypoints in the format in section 1 (one `.npz` per video).
2. Set the paths in section 2.
3. Run all cells. Results land in `results/<RUN_NAME>/results.json`.

Data: `all_videos.txt` anywhere under `dataset/` (labels + signer-independent train/val/test split). The videos themselves aren't needed here.

**Repo layout**
```
dataset/                    labels (all_videos.txt) + videos
keypoints/kp_smplestx/      one .npz per video, per pose model (kp_sapiens/, ...)
results/<run>/              results.json, per_sign_test.csv, confusion_test.npy
sign_classifier.ipynb       this notebook
```

## 1. Input format

One `.npz` per video, named like the video (`13200919064508332-AMAZING.npz`); any folder nesting is fine.

| Key | Shape | Required | Meaning |
| --- | --- | --- | --- |
| `kp2d` | `[T, J, 2]` | for 2D runs | image pixels, x right, y down |
| `kp3d` | `[T, J, 3]` | for 3D runs | any metric 3D frame |
| `conf` | `[T, J]` | optional | 0-1; below `MIN_CONF` counts as missing |
| `joint_names` | `[J]` | recommended | `L_Shoulder` + `R_Shoulder` enable shoulder normalization; `L_`/`R_` pairs enable mirroring |

Missing frames/joints can be `NaN`. For a fair comparison keep a similar joint set to the SMPLest-X runs: upper body (neck, shoulders, elbows, wrists), a few face points (ears, eyes, nose) and 20-21 points per hand, about 50 in total.

```python
np.savez_compressed(out_path, kp2d=kp.astype(np.float32), conf=scores.astype(np.float32),
                    joint_names=np.array(names))
```

## 2. Settings

In [ ]:
import os
# Paths are relative to the repo root (where this notebook lives)
DATA     = "dataset"                  # all_videos.txt is found anywhere inside
KP_DIR   = "keypoints/kp_smplestx"    # e.g. keypoints/kp_sapiens for your own model
COORDS   = "2d"                       # "2d" or "3d"
RUN_NAME = "smplestx_2d"              # results go to results/RUN_NAME - use a new name per model
SEEDS    = 3                          # 3 minimum, 5 for the paper
MIRROR   = False                      # left/right flip augmentation (needs L_/R_ joint names)
MIN_CONF = 0.3
LABEL    = "gloss"                    # "folder" keeps BED1/BED2-style variants apart if they share a gloss

## 3. Install

In [ ]:
!pip install -q scikit-learn

## 4. Classifier code

Written to `sign_classifier.py` so it can also be run from a terminal. Don't edit it per model: changes would break comparability.

In [ ]:
%%writefile sign_classifier.py
"""
sign_classifier.py - isolated sign recognition from keypoint POSITIONS only.

Works with any pose estimator (SMPLest-X, Sapiens, MediaPipe, ...) as long as each video
has a .npz in the shared format:

  kp2d        [T, J, 2]  image pixels            (required for --coords 2d)
  kp3d        [T, J, 3]  any metric 3D frame     (required for --coords 3d)
  conf        [T, J]     optional, 0..1; joints below --min_conf are treated as missing
  joint_names [J]        optional but recommended; "L_Shoulder"/"R_Shoulder" enable
                         shoulder-based normalization, "L_"/"R_" prefixes enable --mirror

Missing joints/frames can be NaN. File name = video file name with .npz instead of .mp4.

Labels and the train/val/test split come from video_info/all_videos.txt
(columns: Participant ID, Video file, Gloss, ASL-LEX Code, Split).

Example:
  python sign_classifier.py --data_root DATASET --kp_dir KP_SMPLESTX --coords 2d --seeds 3 --out_dir results/smplestx_2d
  python sign_classifier.py --data_root DATASET --kp_dir KP_SMPLESTX --coords 3d --seeds 3 --out_dir results/smplestx_3d
  python sign_classifier.py --data_root DATASET --kp_dir KP_SAPIENS  --coords 2d --seeds 3 --out_dir results/sapiens_2d
"""
import argparse
import csv
import json
import os
import os.path as osp
from collections import Counter

import numpy as np
import torch
import torch.nn as nn

T = 64  # every clip is resampled to this many frames


# ------------------------------------------------------------------ preprocessing
def load_keypoints(path, coords, min_conf):
    d = np.load(path, allow_pickle=False)
    key = "kp2d" if coords == "2d" else "kp3d"
    if key not in d.files:
        raise KeyError(f"{path} has no '{key}'")
    kp = d[key].astype(np.float32).copy()                 # [T, J, C]
    if "conf" in d.files:
        kp[d["conf"] < min_conf] = np.nan
    names = [str(n) for n in d["joint_names"]] if "joint_names" in d.files else None
    return kp, names


def normalize(kp, names):
    """Center on the shoulder midpoint and scale by shoulder width (per-video medians,
    so motion relative to the body is kept). Falls back to the mean/std of all points."""
    if names and "L_Shoulder" in names and "R_Shoulder" in names:
        ls, rs = kp[:, names.index("L_Shoulder")], kp[:, names.index("R_Shoulder")]
        center = np.nanmedian((ls + rs) / 2, axis=0)
        scale = np.nanmedian(np.linalg.norm(ls - rs, axis=-1))
    else:
        center = np.nanmean(kp.reshape(-1, kp.shape[-1]), axis=0)
        scale = np.nanstd(kp)
    if not np.isfinite(scale) or scale < 1e-6:
        return None
    return (kp - center) / scale


def fill_and_resample(x, t_out=T):
    """x: [T, D] with NaNs. Interpolate gaps in time, resample to t_out, never-seen dims -> 0."""
    n, D = x.shape
    t = np.arange(n)
    out = np.zeros((t_out, D), np.float32)
    src = np.linspace(0, n - 1, t_out)
    any_ok = False
    for i in range(D):
        ok = ~np.isnan(x[:, i])
        if ok.sum() >= 2:
            out[:, i] = np.interp(src, t[ok], x[ok, i])
            any_ok = True
        elif ok.sum() == 1:
            out[:, i] = x[ok, i][0]
    return out if any_ok else None


def mirror_perm(names):
    """Index permutation that swaps L_* and R_* joints (None if names don't allow it)."""
    if not names:
        return None
    perm = []
    for n in names:
        if n.startswith("L_"):
            m = "R_" + n[2:]
        elif n.startswith("R_"):
            m = "L_" + n[2:]
        else:
            m = n
        if m not in names:
            return None
        perm.append(names.index(m))
    return perm


# ------------------------------------------------------------------ data
def find_list(data_root):
    """Locate all_videos.txt anywhere under data_root (layouts differ between copies)."""
    hits = sorted(
        osp.join(r, f) for r, _, fs in os.walk(data_root) for f in fs
        if f == "all_videos.txt" and "__MACOSX" not in r
    )
    if not hits:
        raise SystemExit(f"all_videos.txt not found under {data_root}")
    if len(hits) > 1:
        print("several all_videos.txt found, using the first:", *hits, sep="\n  ")
    print("labels/splits from", hits[0])
    return hits[0]


def load_dataset(args):
    with open(find_list(args.data_root), newline="", encoding="utf-8") as f:
        rows = [{k.strip(): (v or "").strip() for k, v in r.items()} for r in csv.DictReader(f)]
    files = {}
    for root, _, fs in os.walk(args.kp_dir):
        for fn in fs:
            if fn.endswith(".npz"):
                files[fn[:-4].lower()] = (osp.basename(root), osp.join(root, fn))

    X, labels, split, who = [], [], [], []
    names_ref, missing, bad = None, 0, 0
    for r in rows:
        stem = osp.splitext(r["Video file"])[0].lower()
        if stem not in files:
            missing += 1
            continue
        folder, path = files[stem]
        kp, names = load_keypoints(path, args.coords, args.min_conf)
        if names_ref is None:
            names_ref = names
        kp = normalize(kp, names)
        if kp is None:
            bad += 1
            continue
        x = fill_and_resample(kp.reshape(len(kp), -1))
        if x is None:
            bad += 1
            continue
        X.append(x)
        labels.append(r["Gloss"] if args.label == "gloss" else folder)
        split.append(r["Split"].lower())
        who.append(r["Participant ID"])
    J = X[0].shape[1] // (2 if args.coords == "2d" else 3) if X else 0
    print(f"loaded {len(X)} videos ({J} joints, {args.coords}); {missing} without keypoint file, {bad} unusable")
    if not X:
        raise SystemExit("No usable videos - check --kp_dir and file names.")
    return np.stack(X), np.array(labels), np.array(split), np.array(who), names_ref


def check_splits(split, who):
    sets = {s: set(who[split == s]) for s in ("train", "val", "test")}
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        print(f"participants shared {a}/{b}: {len(sets[a] & sets[b])}")
    for s in sets:
        print(f"  {s}: {(split == s).sum()} videos, {len(sets[s])} participants")


# ------------------------------------------------------------------ model
class SignTransformer(nn.Module):
    def __init__(self, d_in, n_cls, d=192, layers=4, heads=4, drop=0.2):
        super().__init__()
        self.inp = nn.Sequential(nn.Linear(d_in, d), nn.LayerNorm(d), nn.Dropout(drop))
        self.pos = nn.Parameter(torch.zeros(1, T, d))
        enc = nn.TransformerEncoderLayer(d, heads, d * 2, drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(enc, layers)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, n_cls))

    def forward(self, x):
        return self.head(self.enc(self.inp(x) + self.pos).mean(1))


def augment(xb, C, perm=None):
    """xb: [B, T, J*C] normalized positions. Speed/shift, small rotation+scale, optional mirror."""
    B, Tn, D = xb.shape
    J = D // C
    dev = xb.device
    # time: random speed (0.8-1.2x) and shift
    out = torch.empty_like(xb)
    for i in range(B):
        span = (Tn - 1) * min(1.0, np.random.uniform(0.8, 1.2))
        start = np.random.uniform(0, Tn - 1 - span)
        idx = torch.linspace(start, start + span, Tn, device=dev)
        lo = idx.floor().long().clamp(0, Tn - 1)
        hi = (lo + 1).clamp(max=Tn - 1)
        w = (idx - lo).unsqueeze(1)
        out[i] = xb[i, lo] * (1 - w) + xb[i, hi] * w
    p = out.view(B, Tn, J, C)
    # space: rotation about the camera axis (+-10 deg) and scale (0.9-1.1)
    ang = torch.empty(B, device=dev).uniform_(-0.17, 0.17)
    c, s = torch.cos(ang), torch.sin(ang)
    sc = torch.empty(B, device=dev).uniform_(0.9, 1.1)
    x0, y0 = p[..., 0].clone(), p[..., 1].clone()
    p[..., 0] = (c[:, None, None] * x0 - s[:, None, None] * y0) * sc[:, None, None]
    p[..., 1] = (s[:, None, None] * x0 + c[:, None, None] * y0) * sc[:, None, None]
    if C == 3:
        p[..., 2] = p[..., 2] * sc[:, None, None]
    # mirror: flip x and swap left/right joints, for half the batch
    if perm is not None:
        m = torch.rand(B, device=dev) < 0.5
        if m.any():
            q = p[m][:, :, perm].clone()
            q[..., 0] = -q[..., 0]
            p[m] = q
    return p.view(B, Tn, D) + 0.01 * torch.randn_like(out)


def topk(logits, y, k):
    k = min(k, logits.shape[1])
    return float((np.argsort(-logits, 1)[:, :k] == y[:, None]).any(1).mean())


def train_one(Xtr, ytr, Xva, yva, n_cls, C, perm, epochs, seed, dev):
    torch.manual_seed(seed); np.random.seed(seed)
    model = SignTransformer(Xtr.shape[2], n_cls).to(dev)
    steps = epochs * int(np.ceil(len(Xtr) / 32))
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 5e-4, total_steps=steps)
    lossf = nn.CrossEntropyLoss(label_smoothing=0.1)
    Xtr_t, ytr_t, Xva_t = (torch.tensor(a, device=dev) for a in (Xtr, ytr, Xva))
    best, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        order = torch.randperm(len(Xtr_t), device=dev)
        for j in range(0, len(order), 32):
            b = order[j:j + 32]
            loss = lossf(model(augment(Xtr_t[b], C, perm)), ytr_t[b])
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        model.eval()
        with torch.no_grad():
            acc = float((model(Xva_t).argmax(1).cpu().numpy() == yva).mean())
        if acc > best:
            best, best_state = acc, {k: v.detach().clone() for k, v in model.state_dict().items()}
        if (ep + 1) % 50 == 0:
            print(f"    seed {seed} epoch {ep+1}: loss {loss.item():.3f}, val top-1 {acc:.3f} (best {best:.3f})")
    model.load_state_dict(best_state)
    return model.eval()


# ------------------------------------------------------------------ main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", required=True, help="folder with video_info/all_videos.txt")
    ap.add_argument("--kp_dir", required=True, help="folder with keypoint .npz files (any nesting)")
    ap.add_argument("--coords", default="2d", choices=["2d", "3d"])
    ap.add_argument("--label", default="gloss", choices=["gloss", "folder"])
    ap.add_argument("--min_conf", type=float, default=0.3)
    ap.add_argument("--mirror", action="store_true", help="left/right mirror augmentation (needs L_/R_ joint names)")
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--epochs", type=int, default=200)
    ap.add_argument("--out_dir", default="results/run")
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    X, names, split, who, joint_names = load_dataset(args)
    check_splits(split, who)
    classes = sorted(set(names))
    y = np.array([classes.index(n) for n in names])
    C = 2 if args.coords == "2d" else 3
    perm = mirror_perm(joint_names) if args.mirror else None
    if args.mirror and perm is None:
        print("WARNING: --mirror needs matching L_/R_ joint names; mirroring disabled")
    print(f"{len(classes)} classes, input dim {X.shape[2]}, chance {1/len(classes):.3f}")

    tr, va, te = split == "train", split == "val", split == "test"
    # X is already centred on the shoulders and in shoulder-width units. No per-dimension
    # standardisation: it would break the rotation/mirror augmentation geometry.
    mu, g = np.zeros(X.shape[2], np.float32), 1.0
    Xn = X.astype(np.float32)

    res = {"coords": args.coords, "label": args.label, "mirror": perm is not None,
           "n_classes": len(classes), "n_joints": X.shape[2] // C,
           "n_train": int(tr.sum()), "n_val": int(va.sum()), "n_test": int(te.sum()), "runs": []}

    # pooled-feature baseline (no training loop, seconds)
    try:
        from sklearn.linear_model import LogisticRegression
        pool = lambda A: np.concatenate([A.mean(1), A.std(1)], 1)
        lr = LogisticRegression(max_iter=3000, C=0.5).fit(pool(Xn[tr]), y[tr])
        p = lr.predict_proba(pool(Xn[te]))
        res["baseline_test_top1"], res["baseline_test_top5"] = topk(p, y[te], 1), topk(p, y[te], 5)
        print(f"baseline test top-1 {res['baseline_test_top1']:.3f}, top-5 {res['baseline_test_top5']:.3f}")
    except ImportError:
        print("scikit-learn not installed - baseline skipped")

    all_logits = []
    for s in range(args.seeds):
        print(f"training seed {s} ...")
        model = train_one(Xn[tr], y[tr], Xn[va], y[va], len(classes), C, perm, args.epochs, s, dev)
        with torch.no_grad():
            lv = model(torch.tensor(Xn[va], device=dev)).cpu().numpy()
            lt = model(torch.tensor(Xn[te], device=dev)).cpu().numpy()
        run = {"seed": s, "val_top1": topk(lv, y[va], 1), "test_top1": topk(lt, y[te], 1),
               "test_top5": topk(lt, y[te], 5)}
        res["runs"].append(run)
        all_logits.append(lt)
        print(f"  seed {s}: val {run['val_top1']:.3f} | test top-1 {run['test_top1']:.3f}, top-5 {run['test_top5']:.3f}")
        torch.save({"model": model.state_dict(), "classes": classes, "mu": mu, "scale": g,
                    "coords": args.coords, "joint_names": joint_names, "T": T},
                   osp.join(args.out_dir, f"model_seed{s}.pt"))

    for k in ("test_top1", "test_top5", "val_top1"):
        v = np.array([r[k] for r in res["runs"]])
        res[f"{k}_mean"], res[f"{k}_std"] = float(v.mean()), float(v.std())
    ens = np.mean([torch.softmax(torch.tensor(l), 1).numpy() for l in all_logits], 0)
    res["ensemble_test_top1"], res["ensemble_test_top5"] = topk(ens, y[te], 1), topk(ens, y[te], 5)

    # per-sign accuracy and confusions (ensemble)
    pred, yt = ens.argmax(1), y[te]
    cm = np.zeros((len(classes), len(classes)), int)
    for a, b in zip(yt, pred):
        cm[a, b] += 1
    with open(osp.join(args.out_dir, "per_sign_test.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["sign", "n_test", "accuracy", "most_confused_with"])
        for i, c in enumerate(classes):
            row = cm[i].copy(); n = row.sum(); row[i] = -1
            w.writerow([c, n, f"{cm[i, i] / max(n, 1):.3f}", classes[row.argmax()] if n and row.max() > 0 else ""])
    res["top_confusions_test"] = [f"{classes[a]} -> {classes[b]} ({k})" for (a, b), k in
                                  Counter((a, b) for a, b in zip(yt, pred) if a != b).most_common(10)]
    np.save(osp.join(args.out_dir, "confusion_test.npy"), cm)
    with open(osp.join(args.out_dir, "results.json"), "w") as f:
        json.dump(res, f, indent=2)

    print(f"\n== {args.coords}, {res['n_joints']} joints, {args.seeds} seeds ==")
    print(f"test top-1 {res['test_top1_mean']:.3f} +- {res['test_top1_std']:.3f} | "
          f"top-5 {res['test_top5_mean']:.3f} +- {res['test_top5_std']:.3f} | "
          f"ensemble top-1 {res['ensemble_test_top1']:.3f}")


if __name__ == "__main__":
    main()

## 5. Check the keypoint files before training

In [ ]:
import numpy as np, glob
files = glob.glob(os.path.join(KP_DIR, "**", "*.npz"), recursive=True)
print(len(files), "keypoint files")
d = np.load(files[0])
for k in d.files:
    print(f"  {k:12s} {d[k].shape} {d[k].dtype}")
key = "kp2d" if COORDS == "2d" else "kp3d"
assert key in d.files, f"files have no '{key}' - set COORDS correctly"
if "joint_names" in d.files:
    names = [str(n) for n in d["joint_names"]]
    print("joints:", len(names), names[:8], "...")
    print("shoulder normalization:", "L_Shoulder" in names and "R_Shoulder" in names)

## 6. Train and evaluate

In [ ]:
mirror_flag = "--mirror" if MIRROR else ""
!python sign_classifier.py --data_root "{DATA}" --kp_dir "{KP_DIR}" --coords {COORDS} \
    --seeds {SEEDS} --min_conf {MIN_CONF} --label {LABEL} {mirror_flag} --out_dir results/{RUN_NAME}

## 7. Results

In [ ]:
import json, pandas as pd
r = json.load(open(f"results/{RUN_NAME}/results.json"))
print(f"{RUN_NAME}: {r['n_classes']} signs, {r['n_joints']} joints, {r['coords']}")
print(f"test top-1 {r['test_top1_mean']:.3f} +- {r['test_top1_std']:.3f}")
print(f"test top-5 {r['test_top5_mean']:.3f} +- {r['test_top5_std']:.3f}")
print(f"seed ensemble top-1 {r['ensemble_test_top1']:.3f}")
print("most common confusions:", *r["top_confusions_test"][:5], sep="\n  ")
pd.read_csv(f"results/{RUN_NAME}/per_sign_test.csv").sort_values("accuracy").head(10)

## 8. Compare all runs

Lists every run in `results/`, so put everyone's `results.json` folders side by side here.

In [ ]:
rows = []
for p in sorted(glob.glob("results/*/results.json")):
    r = json.load(open(p))
    if "test_top1_mean" not in r:
        continue
    rows.append({"run": os.path.basename(os.path.dirname(p)), "coords": r["coords"], "joints": r["n_joints"],
                 "seeds": len(r["runs"]), "top-1": f"{r['test_top1_mean']:.3f} ± {r['test_top1_std']:.3f}",
                 "top-5": f"{r['test_top5_mean']:.3f} ± {r['test_top5_std']:.3f}",
                 "ensemble top-1": round(r["ensemble_test_top1"], 3)})
pd.DataFrame(rows)

## Appendix: SMPLest-X keypoints

Only needed to regenerate the SMPLest-X keypoints from raw SMPLest-X outputs (`batch_extract.py`). Produces both `kp2d` and `kp3d` on the same 52 joints.

In [ ]:
%%writefile smplestx_to_keypoints.py
"""
smplestx_to_keypoints.py - convert SMPLest-X .npz files (from batch_extract.py) into the
shared keypoint format used by sign_classifier.py.

For each video it writes one .npz with:
  kp2d        [T, J, 2]  image pixels (x right, y down), from SMPLest-X's 2D projection
  kp3d        [T, J, 3]  metres, root-relative (pelvis), camera axes (x right, y down, z forward)
  conf        [T, J]     1.0 where the person was detected, 0.0 otherwise
  joint_names [J]        names; L_/R_ prefixes are used for mirroring and normalization
  image_hw, video_fps

Joint set (J = 52): 12 upper-body/head + 20 left-hand + 20 right-hand points.

Usage (no GPU needed):
  python smplestx_to_keypoints.py --params_dir PARAMS --out_dir KP_SMPLESTX
"""
import argparse
import os
import os.path as osp

import numpy as np

# SMPLest-X 137-joint order: body 0-24, left hand 25-44, right hand 45-64, face 65-136
BODY_NAMES = ["Pelvis", "L_Hip", "R_Hip", "L_Knee", "R_Knee", "L_Ankle", "R_Ankle", "Neck",
              "L_Shoulder", "R_Shoulder", "L_Elbow", "R_Elbow", "L_Wrist", "R_Wrist",
              "L_Big_toe", "L_Small_toe", "L_Heel", "R_Big_toe", "R_Small_toe", "R_Heel",
              "L_Ear", "R_Ear", "L_Eye", "R_Eye", "Nose"]
FINGERS = ["Thumb", "Index", "Middle", "Ring", "Pinky"]
HAND_NAMES = {s: [f"{s}_{f}_{i}" for f in FINGERS for i in range(1, 5)] for s in ("L", "R")}
ALL_NAMES = BODY_NAMES + HAND_NAMES["L"] + HAND_NAMES["R"]

KEEP_BODY = ["Neck", "L_Shoulder", "R_Shoulder", "L_Elbow", "R_Elbow", "L_Wrist", "R_Wrist",
             "L_Ear", "R_Ear", "L_Eye", "R_Eye", "Nose"]
KEEP = KEEP_BODY + HAND_NAMES["L"] + HAND_NAMES["R"]
KEEP_IDX = [ALL_NAMES.index(n) for n in KEEP]

# SMPLest-X constants (configs: input_body_shape = (256, 192), output_hm_shape = (16, 16, 12)).
# smplx_joint_proj is in output-heatmap coordinates of the person crop.
HM_W, HM_H = 12.0, 16.0


def convert(d):
    n = len(d["frame"])
    det = d["detected"].astype(bool)
    bbox = d["bbox_xywh"]                                   # crop box in image pixels
    proj = d["smplx_joint_proj"][:, KEEP_IDX]               # [T, J, 2] heatmap coords
    kp2d = np.empty_like(proj)
    kp2d[..., 0] = proj[..., 0] / HM_W * bbox[:, None, 2] + bbox[:, None, 0]
    kp2d[..., 1] = proj[..., 1] / HM_H * bbox[:, None, 3] + bbox[:, None, 1]
    kp3d = d["smplx_joint_cam"][:, KEEP_IDX]
    conf = np.repeat(det[:, None].astype(np.float32), len(KEEP), 1)
    kp2d[~det] = np.nan
    kp3d[~det] = np.nan
    return {"kp2d": kp2d.astype(np.float32), "kp3d": kp3d.astype(np.float32), "conf": conf,
            "joint_names": np.array(KEEP), "frame": d["frame"],
            "image_hw": d["image_hw"] if "image_hw" in d.files else np.array([0, 0]),
            "video_fps": d["video_fps"]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--params_dir", required=True)
    ap.add_argument("--out_dir", required=True)
    args = ap.parse_args()

    n_ok = n_skip = 0
    for root, _, files in os.walk(args.params_dir):
        for fn in files:
            if not fn.endswith(".npz"):
                continue
            d = np.load(osp.join(root, fn))
            if "smplx_joint_proj" not in d.files:
                n_skip += 1
                continue
            out = osp.join(args.out_dir, osp.relpath(root, args.params_dir), fn)
            os.makedirs(osp.dirname(out), exist_ok=True)
            np.savez_compressed(out, **convert(d))
            n_ok += 1
    print(f"converted {n_ok} videos, skipped {n_skip} (no person detected) -> {args.out_dir}")


if __name__ == "__main__":
    main()

In [ ]:
# PARAMS = "/path/to/smplestx/params"   # raw outputs of batch_extract.py
# !python smplestx_to_keypoints.py --params_dir {PARAMS} --out_dir {KP_DIR}